# Poster NeRF — Hierarchical Sampling

This notebook trains a NeRF on the Poster dataset using hierarchical coarse-to-fine sampling. It preserves the original Poster camera pipeline and MLP architecture, with separate coarse and fine networks.


In [ ]:
from pathlib import Path  
import json  


import cv2   
import numpy as np 
import matplotlib.pyplot as plt
from PIL import Image



import torch    
from pytorch3d.renderer import (
    PerspectiveCameras,
    MonteCarloRaysampler,
    NDCMultinomialRaysampler,
    EmissionAbsorptionRaymarcher,
    RayBundle,
    ray_bundle_to_ray_points,
)



device = torch.device("cuda")   
print(torch.cuda.get_device_name(0))  

## Data

The full-resolution Poster images (1080 × 1920) and camera poses are loaded from `data/poster`.


In [ ]:
data_dir = Path("data/poster")   
image_folder = "images"   


output_dir = Path("poster_outputs_hierarchical")   
output_dir.mkdir(exist_ok=True)
with (data_dir / "transforms.json").open() as file:   
    transforms = json.load(file)   

frames = sorted(transforms["frames"], key=lambda frame: frame["file_path"])   
frame_ids = [Path(frame["file_path"]).name for frame in frames]   


print("Images:", len(frames))  


for folder in ["images", "images_2", "images_4", "images_8"]:    
    with Image.open(data_dir / folder / frame_ids[0]) as image:    
        print(folder, image.size)


The dataset loader uses Poster's camera intrinsics and poses from `transforms.json`. The saved poses already include `applied_transform`.


In [ ]:
def load_nerf_data(data_dir, image_folder):   
    with open(data_dir / "transforms.json") as f:   
        transforms_data = json.load(f)  

     
    frames = sorted(transforms_data["frames"], key=lambda frame: frame["file_path"])   
    
    first_image = data_dir / image_folder / Path(frames[0]["file_path"]).name  
    with Image.open(first_image) as img:   
        width, height = img.size  
     
  
    # Keep one copy of the images in RAM   
    target_images = torch.empty(   
        (len(frames), height, width, 3),
        dtype=torch.float32,   
    )   
   
  
   
    Rs, Ts = [], []    
    axis_change = torch.diag(torch.tensor([-1.0, 1.0, -1.0]))   
     
  
    for index, frame in enumerate(frames):
        img_path = data_dir / image_folder / Path(frame["file_path"]).name    
     
        with Image.open(img_path) as img:    
            img_np = np.array(img).astype(np.float32) / 255.0    
   
        target_images[index] = torch.from_numpy(img_np)   
   
    
        c2w = torch.tensor(frame["transform_matrix"], dtype=torch.float32)   
        R_nerf = c2w[:3, :3]  
        T_nerf = c2w[:3, 3]   

        # Convert the saved camera axes to PyTorch3D    
        R_pytorch3d = R_nerf @ axis_change   
        T_pytorch3d = -T_nerf @ R_pytorch3d  
        Rs.append(R_pytorch3d)  
        Ts.append(T_pytorch3d)  



    scale_x = width / transforms_data["w"]   
    scale_y = height / transforms_data["h"]   

 
    target_cameras = PerspectiveCameras( 
        R=torch.stack(Rs),    
        T=torch.stack(Ts),   
        focal_length=((    
            transforms_data["fl_x"] * scale_x,  
            transforms_data["fl_y"] * scale_y, 
        ),),
        principal_point=((
            transforms_data["cx"] * scale_x,
            transforms_data["cy"] * scale_y,
        ),),
        image_size=((height, width),),
        in_ndc=False,
    )
   
    return target_images, target_cameras   

In [ ]:
target_images, target_cameras = load_nerf_data(data_dir, image_folder)   
image_height, image_width = target_images.shape[1:3]  
     


# Every tenth image is kept for testing    
indices = torch.arange(len(frames)) 
test_indices = indices[indices % 10 == 1]  
train_indices = indices[indices % 10 != 1]  

print("Train:", len(train_indices), "Test:", len(test_indices))   
  

   

fig, axes = plt.subplots(1, 3, figsize=(8, 5))   

for ax, index in zip(axes, [0, 112, 225]):  
    ax.imshow(target_images[index])   
    ax.set_title(frame_ids[index])
    ax.axis("off")  
   

plt.tight_layout()   
plt.show()   

## Model

Each NeRF uses the original two-layer MLP with harmonic embeddings for position and viewing direction. Density is converted to opacity using the distance between ray samples.


In [ ]:
class HarmonicEmbedding(torch.nn.Module):    

    def __init__(self, n_harmonic_functions):    

        super().__init__()   
        self.register_buffer(  
            'frequencies',   
            2.0 ** torch.arange(n_harmonic_functions),   
        )  


    def forward(self, x):   
        embed = (x[..., None] * self.frequencies).view(*x.shape[:-1], -1)  
        return torch.cat((x, embed.sin(), embed.cos()), dim=-1)   
  
     


class NeuralRadianceField(torch.nn.Module):  
    def __init__(self, n_hidden_neurons=256):    
        super().__init__()    
   
        self.position_embedding = HarmonicEmbedding(10)   
        self.direction_embedding = HarmonicEmbedding(4)   
  
        position_dim = 3 + 10 * 2 * 3  
        direction_dim = 3 + 4 * 2 * 3
  

        self.mlp = torch.nn.Sequential(    
            torch.nn.Linear(position_dim, n_hidden_neurons),
            torch.nn.Softplus(beta=10.0),
            torch.nn.Linear(n_hidden_neurons, n_hidden_neurons),   
            torch.nn.Softplus(beta=10.0),
        )



        self.color_layer = torch.nn.Sequential(      
            torch.nn.Linear(n_hidden_neurons + direction_dim, n_hidden_neurons),
            torch.nn.Softplus(beta=10.0),
            torch.nn.Linear(n_hidden_neurons, 3),
            torch.nn.Sigmoid(),

        )  


        self.density_layer = torch.nn.Sequential(    
            torch.nn.Linear(n_hidden_neurons, 1),  
            torch.nn.Softplus(beta=10.0),   

        )

        self.density_layer[0].bias.data[0] = -1.5    

    def _get_densities(self, features, ray_bundle):   
        raw_densities = self.density_layer(features)    

   
        depths = ray_bundle.lengths  
        intervals = depths[..., 1:] - depths[..., :-1]  
        last_interval = max_depth - depths[..., -1:]   
        intervals = torch.cat((intervals, last_interval), dim=-1)   

        # Depth is measured along camera z, include the ray direction length   
        distances = intervals * ray_bundle.directions.norm(dim=-1, keepdim=True)

        return 1 - (-raw_densities * distances[..., None]).exp()    

    def _get_colors(self, features, rays_directions):   
        spatial_size = features.shape[:-1]  
  
        rays_directions_normed = torch.nn.functional.normalize( 
            rays_directions, dim=-1   
        )  

        rays_embedding = self.direction_embedding(  
            rays_directions_normed  
        )

        rays_embedding_expand = rays_embedding[..., None, :].expand(  
            *spatial_size, rays_embedding.shape[-1]   
        )

        color_layer_input = torch.cat(
            (features, rays_embedding_expand),     
            dim=-1
        )
        return self.color_layer(color_layer_input)   



    def forward(self, ray_bundle):   

        rays_points_world = ray_bundle_to_ray_points(ray_bundle)  

        embeds = self.position_embedding( 
            rays_points_world
        )

        features = self.mlp(embeds)

 
        rays_densities = self._get_densities(features, ray_bundle)  

        rays_colors = self._get_colors(features, ray_bundle.directions)  

        return rays_densities, rays_colors   

### Hierarchical Sampling

A coarse network samples 64 depths per ray and guides 128 additional importance samples. The fine network evaluates all 192 depths.


In [ ]:
def alpha_to_weights(alpha):
    """Convert opacity to volume-rendering weights."""
    a = alpha[..., 0]
    transmittance = torch.cumprod(
        torch.cat([torch.ones_like(a[..., :1]), 1 - a + 1e-10], dim=-1),
        dim=-1,
    )[..., :-1]
    return a * transmittance


def sample_pdf(bins, weights, n_samples, perturb):
    """Sample depths from a piecewise-constant PDF."""
    leading_dims = bins.shape[:-1]
    bins = bins.reshape(-1, bins.shape[-1])
    weights = weights.reshape(-1, weights.shape[-1])

    stabilized = weights + 1e-5
    pdf = stabilized / stabilized.sum(dim=-1, keepdim=True)
    cdf = torch.cat(
        [torch.zeros_like(pdf[..., :1]), torch.cumsum(pdf, dim=-1)],
        dim=-1,
    )

    if perturb:
        u = torch.rand(cdf.shape[0], n_samples, device=cdf.device, dtype=cdf.dtype)
    else:
        u = torch.linspace(
            0., 1. - 1e-6, n_samples, device=cdf.device, dtype=cdf.dtype
        ).expand(cdf.shape[0], n_samples).contiguous()

    indices = torch.searchsorted(cdf.contiguous(), u.contiguous(), right=True)
    lower = (indices - 1).clamp(min=0)
    upper = indices.clamp(max=cdf.shape[-1] - 1)
    gather_indices = torch.stack([lower, upper], dim=-1)

    adjacent_cdf = torch.gather(
        cdf[:, None, :].expand(-1, n_samples, -1), 2, gather_indices
    )
    adjacent_bins = torch.gather(
        bins[:, None, :].expand(-1, n_samples, -1), 2, gather_indices
    )

    delta_cdf = adjacent_cdf[..., 1] - adjacent_cdf[..., 0]
    delta_cdf = torch.where(
        delta_cdf < 1e-5, torch.ones_like(delta_cdf), delta_cdf
    )
    samples = adjacent_bins[..., 0] + (
        (u - adjacent_cdf[..., 0]) / delta_cdf
    ) * (adjacent_bins[..., 1] - adjacent_bins[..., 0])
    return samples.reshape(*leading_dims, n_samples)


class HierNeRF(torch.nn.Module):
    """Coarse and fine NeRF networks."""

    def __init__(self, coarse, fine, n_fine):
        super().__init__()
        self.coarse = coarse
        self.fine = fine
        self.n_fine = n_fine
        self.aux = {}

    def forward(self, rays):
        alpha_coarse, color_coarse = self.coarse(rays)
        coarse_weights = alpha_to_weights(alpha_coarse)
        depths = rays.lengths

        midpoints = 0.5 * (depths[..., 1:] + depths[..., :-1])
        extra_depths = sample_pdf(
            midpoints, coarse_weights[..., 1:-1].detach(),
            self.n_fine, perturb=self.training,
        ).detach()

        all_depths = torch.sort(
            torch.cat([depths, extra_depths], dim=-1), dim=-1
        ).values
        fine_rays = RayBundle(
            origins=rays.origins,
            directions=rays.directions,
            lengths=all_depths,
            xys=rays.xys,
        )
        alpha_fine, color_fine = self.fine(fine_rays)
        self.aux = {
            "coarse": (alpha_coarse, color_coarse),
            "fine_lengths": all_depths,
        }
        return alpha_fine, color_fine


## Training

Train for 20,000 iterations with three images and 750 rays per image. Both coarse and fine RGB predictions use MSE loss. Adam starts at 0.001, with a tenfold learning-rate reduction after 15,000 iterations.


In [ ]:
n_iter = 20000    
batch_size = 3    
lr = 1e-3    


N_COARSE = 64
N_FINE = 128
samples_per_ray = N_COARSE    
min_depth = 0.01   
max_depth = 32.0  



range_x = image_width / min(image_height, image_width)    
range_y = image_height / min(image_height, image_width)   


raysampler_mc = MonteCarloRaysampler(     
    min_x=-range_x + range_x / image_width,   
    max_x=range_x - range_x / image_width,  
    min_y=-range_y + range_y / image_height,
    max_y=range_y - range_y / image_height,  
    n_rays_per_image=750,   
    n_pts_per_ray=samples_per_ray,  
    min_depth=min_depth,   
    max_depth=max_depth,  
    stratified_sampling=True, 
).to(device)     

raymarcher = EmissionAbsorptionRaymarcher()   



scale_x = image_width / transforms["w"]   
scale_y = image_height / transforms["h"]   

camera_matrix = np.array([   
    [transforms["fl_x"] * scale_x, 0, transforms["cx"] * scale_x],   
    [0, transforms["fl_y"] * scale_y, transforms["cy"] * scale_y], 
    [0, 0, 1],
])


distortion_coefficients = np.array([   
    transforms["k1"], transforms["k2"],    
    transforms["p1"], transforms["p2"],    
])









def correct_ray_directions(rays, cameras):    
    pixels = rays.xys.detach().cpu().numpy().reshape(-1, 2).copy()   
    ndc_scale = min(image_height, image_width) / 2
    pixels[:, 0] = image_width / 2 - pixels[:, 0] * ndc_scale   
    pixels[:, 1] = image_height / 2 - pixels[:, 1] * ndc_scale


    ideal = cv2.undistortPoints(    

        pixels[:, None, :],    
        camera_matrix,
        distortion_coefficients,
    )[:, 0, :]


    camera_directions = np.column_stack((-ideal, np.ones(len(ideal))))   
    camera_directions = camera_directions.astype(np.float32)  
    camera_directions = torch.from_numpy(camera_directions).to(device)      
    camera_directions = camera_directions.reshape(len(cameras), -1, 3)   

    world_directions = torch.bmm(camera_directions, cameras.R.transpose(1, 2))   



    # Keep the original pixel coordinates for the target colours    
    return RayBundle(
        origins=rays.origins,
        directions=world_directions.reshape_as(rays.directions),
        lengths=rays.lengths,
        xys=rays.xys,
    )








def sample_rgb(target_images, sampled_rays_xy):     
    num_images = target_images.shape[0] 
    num_channels = target_images.shape[-1]   
    spatial_size = sampled_rays_xy.shape[1:-1]  


    height, width = target_images.shape[1:3]
    ranges = target_images.new_tensor([    
        width / min(height, width),
        height / min(height, width),   
    ])



    # Scale rectangular image coordinates to [-1, 1] for grid_sample    
    sampled_rays_xy = sampled_rays_xy / ranges   

    images_sampled = torch.nn.functional.grid_sample(   
        target_images.permute(0, 3, 1, 2),    
        -sampled_rays_xy.view(num_images, -1, 1, 2),    
        align_corners=False   
    )
    return images_sampled.permute(0, 2, 3, 1).view(   
        num_images, *spatial_size, num_channels   
    )


In [ ]:
torch.manual_seed(1)
model = HierNeRF(
    coarse=NeuralRadianceField(),
    fine=NeuralRadianceField(),
    n_fine=N_FINE,
).to(device)




optimizer = torch.optim.Adam(
    model.parameters(),
    lr=lr,
)
loss_history = []
coarse_loss_history = []
total_loss_history = []



for iteration in range(n_iter):
    if iteration == round(n_iter * 0.75):
        for group in optimizer.param_groups:
            group["lr"] = lr * 0.1


    optimizer.zero_grad()
    random_order = torch.randperm(len(train_indices))
    batch_idx = train_indices[random_order[:batch_size]]
    batch_cameras = target_cameras[batch_idx].to(device)

    sampled_rays = raysampler_mc(cameras=batch_cameras)
    sampled_rays = correct_ray_directions(sampled_rays, batch_cameras)


    opacity_fine, colour_fine = model(sampled_rays)
    rendered_fine = raymarcher(
        rays_densities=opacity_fine, rays_features=colour_fine
    )[..., :3]

    opacity_coarse, colour_coarse = model.aux["coarse"]
    rendered_coarse = raymarcher(
        rays_densities=opacity_coarse, rays_features=colour_coarse
    )[..., :3]

    colors_at_rays = sample_rgb(
        target_images[batch_idx].to(device), sampled_rays.xys
    )
    loss_fine = torch.nn.functional.mse_loss(rendered_fine, colors_at_rays)
    loss_coarse = torch.nn.functional.mse_loss(rendered_coarse, colors_at_rays)
    loss = loss_coarse + loss_fine

    loss_history.append(float(loss_fine.detach()))
    coarse_loss_history.append(float(loss_coarse.detach()))
    total_loss_history.append(float(loss.detach()))


    loss.backward()
    optimizer.step()


    if (iteration + 1) % 100 == 0:
        print(
            f"step {iteration+1}/{n_iter} | fine MSE {loss_history[-1]:.6f} "
            f"| coarse MSE {coarse_loss_history[-1]:.6f}"
        )







torch.save(model.state_dict(), output_dir / "model.pt")

np.savez_compressed(
    output_dir / "loss_history.npz",
    total=total_loss_history,
    fine=loss_history,
    coarse=coarse_loss_history,
)
print("Saved hierarchical model and loss histories to:", output_dir)


In [ ]:
plt.plot(range(1, n_iter + 1), loss_history, label="Fine RGB MSE")
plt.plot(range(1, n_iter + 1), coarse_loss_history, label="Coarse RGB MSE", alpha=0.65)
plt.legend()     



plt.xlabel("Iteration")   
plt.ylabel("Training RGB MSE")  




plt.tight_layout()   
plt.savefig(output_dir / "loss.png", dpi=150)   
plt.show()    

## Test Images

Render the 23 held-out Poster views and save predictions and targets for evaluation in `3-eval.ipynb`.


In [ ]:
raysampler_grid = NDCMultinomialRaysampler(   
    image_height=image_height,   
    image_width=image_width,
    n_pts_per_ray=samples_per_ray,
    min_depth=min_depth,   
    max_depth=max_depth,
    stratified_sampling=False,  
).to(device)  
   




  

  

def render_image(camera):   
    with torch.no_grad():  
        rays = raysampler_grid(cameras=camera)
        rays = correct_ray_directions(rays, camera)  
  
    
        origins = rays.origins.reshape(-1, 3)   
        directions = rays.directions.reshape(-1, 3)  
        lengths = rays.lengths.reshape(-1, samples_per_ray)  
        xy = rays.xys.reshape(-1, 2)    

  
        image = torch.empty((image_height * image_width, 3))   



    
        for start in range(0, len(image), 1024):
            stop = start + 1024      
            chunk = RayBundle(
                origins=origins[None, start:stop],
                directions=directions[None, start:stop],  
                lengths=lengths[None, start:stop],   
                xys=xy[None, start:stop], 
                     
            )


            opacity, colour = model(chunk)   
            rendered = raymarcher(rays_densities=opacity, rays_features=colour)  
            image[start:stop] = rendered[0, :, :3].cpu()  



    return image.reshape(image_height, image_width, 3).clamp(0, 1)    

In [ ]:
model.eval()

predictions = []
test_names = [frame_ids[index] for index in test_indices.tolist()]

for index in test_indices.tolist():
    image = render_image(target_cameras[index].to(device))
    predictions.append(image.numpy())
    print(frame_ids[index])

predictions = np.stack(predictions)
targets = target_images[test_indices].numpy()

np.savez_compressed(
    output_dir / "poster_test_images.npz",
    predictions=predictions,
    targets=targets,
    frame_ids=np.array(test_names),
)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 9))

for column, position in enumerate([0, 11, 22]):
    axes[0, column].imshow(targets[position])
    axes[0, column].set_title(test_names[position])

    axes[1, column].imshow(predictions[position])
    axes[1, column].set_title("Prediction")

    axes[0, column].axis("off")
    axes[1, column].axis("off")

plt.tight_layout()
plt.savefig(output_dir / "comparison.png", dpi=150)
plt.show()


## Evaluation

The saved `poster_test_images.npz` can be evaluated in `3-eval.ipynb` using MSE, PSNR, SSIM and LPIPS. Training results and test renders are saved to `poster_outputs_hierarchical`.
